# G4 — Stage 34b XAI: Faithfulness + Stability

Stage 34b: ProtoSegNet L3+L4, with skip, ALC L3-only (λ=0.05)  
Checkpoint: `proto_seg_ct_l3l4_alc_l3only.pth`  
Projection norms consistent → projection file is valid, no re-projection needed.

Already known from v8: AP=0.221, Purity=0.593  
Missing: Faithfulness, Stability, Patch Faithfulness

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, NUM_CLASSES
from src.models.proto_seg_net import ProtoSegNet
from src.metrics.faithfulness import faithfulness_patient
from src.metrics.stability import stability_patient
from src.metrics.patch_faithfulness import patch_faithfulness_patient

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_SLICES = 50

class Adapter(nn.Module):
    """Wrap ProtoSegNet (2-output) to 3-output interface."""
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x, **kw):
        logits, hm = self.base(x)
        n = len(self.base.proto_levels)
        w = torch.full((x.size(0), n), 1.0/n, device=x.device)
        return logits, hm, w

print(f'Device: {DEVICE}')

Device: mps


## 1. Load Stage 34b

In [2]:
CKPT  = CKPT_DIR / 'proto_seg_ct_l3l4_alc_l3only.pth'
PROJ  = CKPT_DIR / 'projected_prototypes_ct_l3l4_alc_l3only.pt'

ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
print(f'Checkpoint: epoch={ckpt["epoch"]}  best_val={ckpt["best_val_dice"]:.4f}')

model = ProtoSegNet(
    n_classes=NUM_CLASSES,
    proto_levels=[3, 4],
    use_level_attention=ckpt.get('use_level_attention', False),
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

# Projection norms match checkpoint — safe to load
proj = torch.load(PROJ, map_location='cpu', weights_only=True)
for level, proto_data in proj['proto_state'].items():
    model.proto_layers[str(level)].prototypes.data.copy_(proto_data)
print(f'Projection loaded from {PROJ.name}')
model.eval()

adapted = Adapter(model).to(DEVICE)
adapted.eval()
print('Ready.')

Checkpoint: epoch=75  best_val=0.8416
Projection loaded from projected_prototypes_ct_l3l4_alc_l3only.pt
Ready.


## 2. Test images

In [3]:
test_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'test', augment=False, preload=True)
imgs_test = torch.stack([test_ds[i]['image'] for i in range(len(test_ds))])
print(f'Test images: {tuple(imgs_test.shape)}')

Test images: (484, 1, 256, 256)


## 3. Faithfulness (pixel-level)

In [4]:
print('Computing Faithfulness (50 slices)…')
faith = faithfulness_patient(adapted, imgs_test, DEVICE, max_slices=MAX_SLICES)
print(f'  Faithfulness : {faith["faithfulness"]:.4f}  (std {faith["faithfulness_std"]:.4f})')
pd.DataFrame([faith]).to_csv(OUT_DIR / 'xai_faithfulness_34b.csv', index=False)

Computing Faithfulness (50 slices)…


  Faithfulness : 0.0833  (std 0.0371)


## 4. Stability

In [5]:
print('Computing Stability (50 slices)…')
stab = stability_patient(adapted, imgs_test, DEVICE, max_slices=MAX_SLICES)
print(f'  Stability    : {stab["stability"]:.4f}  (std {stab["stability_std"]:.4f})')
pd.DataFrame([stab]).to_csv(OUT_DIR / 'xai_stability_34b.csv', index=False)

Computing Stability (50 slices)…


  Stability    : 9.2361  (std 1.1376)


## 5. Patch Faithfulness (bs=16)

In [6]:
print('Computing Patch Faithfulness (block_size=16, 50 slices)…')
pfaith = patch_faithfulness_patient(adapted, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  Patch Faith  : {pfaith["patch_faithfulness"]:.4f}  (std {pfaith["patch_faithfulness_std"]:.4f})')
pd.DataFrame([pfaith]).to_csv(OUT_DIR / 'xai_patch_faithfulness_34b.csv', index=False)

Computing Patch Faithfulness (block_size=16, 50 slices)…


  Patch Faith  : 0.3015  (std 0.1080)


## 6. Summary

In [7]:
# Known from v8 results
val_dice = ckpt['best_val_dice']
eff_pur  = 0.593   # from results/v8/effective_quality_ct_l3l4_alc_l3only.csv
eff_ap   = 0.221   # same
faith_v  = faith['faithfulness']
stab_v   = stab['stability']
pfaith_v = pfaith['patch_faithfulness']

rows = [
    ('Val Dice',           val_dice, None, None,  '—'),
    ('Effective Purity',   eff_pur,  None, None,  '—'),
    ('Effective AP',       eff_ap,   0.15, 0.25,  '✅' if eff_ap   >= 0.15 else '❌'),
    ('Faithfulness (px)',  faith_v,  0.15, 0.30,  '✅' if faith_v  >= 0.15 else '❌'),
    ('Stability',          stab_v,   None, 2.00,  '✅' if stab_v   <= 2.00 else '❌'),
    ('Patch Faith (bs16)', pfaith_v, None, None,  '—'),
]
summary = pd.DataFrame(rows, columns=['Metric', 'Value', 'Min gate', 'Target', 'Pass'])
summary.to_csv(OUT_DIR / 'xai_summary_34b.csv', index=False)

print('Stage 34b (skip, L3+L4, ALC L3-only) — XAI Summary')
print('=' * 60)
print(summary.to_string(index=False))
print()

# Compare to Stage 29 and 9b
s29_faith = 0.069; s29_stab = 3.382; s29_ap = 0.051; s29_dice = 0.821
s9b_faith = 0.035; s9b_stab = 11.94; s9b_ap  = 0.301; s9b_dice = 0.559

print('Context — skip model family comparison:')
print(f'{"Model":<25} {"Dice":>6} {"AP":>6} {"Faith":>7} {"Stab":>7} {"PFaith":>8}')
print('-'*60)
print(f'{"Stage 8 Full (L1-4, skip)":<25} {0.817:>6.3f} {0.102:>6.3f} {0.059:>7.3f} {3.00:>7.2f} {"—":>8}')
print(f'{"Stage 34b (L3+4, ALC)":<25} {val_dice:>6.3f} {eff_ap:>6.3f} {faith_v:>7.3f} {stab_v:>7.2f} {pfaith_v:>8.3f}')
print(f'{"Stage 29 (L3+4, warmst.)":<25} {s29_dice:>6.3f} {s29_ap:>6.3f} {s29_faith:>7.3f} {s29_stab:>7.2f} {0.212:>8.3f}')
print(f'{"9b (L3+4, no-skip)":<25} {s9b_dice:>6.3f} {s9b_ap:>6.3f} {s9b_faith:>7.3f} {s9b_stab:>7.2f} {0.200:>8.3f}')

Stage 34b (skip, L3+L4, ALC L3-only) — XAI Summary
            Metric    Value  Min gate  Target Pass
          Val Dice 0.841584       NaN     NaN    —
  Effective Purity 0.593000       NaN     NaN    —
      Effective AP 0.221000      0.15    0.25    ✅
 Faithfulness (px) 0.083255      0.15    0.30    ❌
         Stability 9.236069       NaN    2.00    ❌
Patch Faith (bs16) 0.301519       NaN     NaN    —

Context — skip model family comparison:
Model                       Dice     AP   Faith    Stab   PFaith
------------------------------------------------------------
Stage 8 Full (L1-4, skip)  0.817  0.102   0.059    3.00        —
Stage 34b (L3+4, ALC)      0.842  0.221   0.083    9.24    0.302
Stage 29 (L3+4, warmst.)   0.821  0.051   0.069    3.38    0.212
9b (L3+4, no-skip)         0.559  0.301   0.035   11.94    0.200
